# QAHERRA reproduction notebook

Runs the reproduction steps in Google Colab against the archived artifacts.
Each cell is independent and saves its output to Drive before returning, so an
interrupted session resumes without repeating work.

**Credentials.** No API key is stored in this notebook. Set it in the cell
below, or better, read it from a Colab secret. A key pasted into a notebook
that is later shared is a key that has to be revoked.

Order of cells:

1. install and clone
2. mount Drive and set paths
3. verify every reported number from the saved per-question files (no API calls)
4. the corrected four-document fixed-size chunk baselines
5. the paired tests for those baselines
6. deduplication stages A to D
7. Figure 3


## 1. Install and clone

In [ ]:
!pip -q install bm25s==0.2.5 faiss-cpu sentence-transformers openai json5 scipy statsmodels psutil matplotlib
import os
if not os.path.isdir('/content/QAHERRA'):
    !git clone -q https://github.com/Sardar-Jaffar-Ali/QAHERRA /content/QAHERRA
!cd /content/QAHERRA && pip install -q -e . --no-deps 2>&1 | tail -1
print('ready')

## 2. Mount Drive and set paths

If a previous attempt created `/content/drive` as an ordinary folder, mounting
fails with `Mountpoint must not already contain files`. Removing the folder
first avoids that.

In [ ]:
import shutil, os
shutil.rmtree('/content/drive', ignore_errors=True)
from google.colab import drive
drive.mount('/content/drive')

# Read the key from the environment or a Colab secret; do not paste it here.
os.environ.setdefault('OPENAI_API_KEY', 'PASTE_YOUR_API_KEY_HERE')
assert os.environ['OPENAI_API_KEY'] != 'PASTE_YOUR_API_KEY_HERE', 'set the API key before any cell that generates answers'

# configs/paths.yaml maps every artifact to its location on this Drive.
!cd /content/QAHERRA && cp -n configs/paths.example.yaml configs/paths.yaml
print('edit /content/QAHERRA/configs/paths.yaml, then continue')

## 3. Verify every reported number

No API calls, no index, no GPU. Reads the saved per-question files, recomputes
each accuracy, each paired test, and each context-length figure, and compares
them with the values printed in the paper.

In [ ]:
!cd /content/QAHERRA && python scripts/verify_results.py --paths configs/paths.yaml

## 4. The corrected four-document chunk baselines

The chunk text is read from the four-document index metadata. The run asserts
that SHA-1 of every chunk's text reproduces its recorded hash before generating
anything; see `docs/CHUNK_LOOKUP_FIX.md`. A failed API call is logged and never
scored, so rerunning a cell retries only the questions that failed.

In [ ]:
!cd /content/QAHERRA && python scripts/run_chunk_baseline_4doc.py --retriever bm25 --paths configs/paths.yaml

In [ ]:
!cd /content/QAHERRA && python scripts/run_chunk_baseline_4doc.py --retriever dense --paths configs/paths.yaml

## 5. Paired tests for those baselines

Expected: BM25 84 against 66, 22 against 4 discordant, Holm-corrected
p = 0.003; Dense 74 against 60, 24 against 10, Holm-corrected p = 0.073 within
the declared family of five.

In [ ]:
!cd /content/QAHERRA && python scripts/verify_results.py --paths configs/paths.yaml --write-results

## 6. Deduplication, stages A to D

Stage A takes about six minutes and roughly 12 GB of RAM, and needs no API
calls. Stage D is the only stage that generates answers. Each stage checks its
inputs and stops without recording a result if a check fails.

In [ ]:
!cd /content/QAHERRA && python scripts/dedup_stage_a_build.py --paths configs/paths.yaml

In [ ]:
!cd /content/QAHERRA && python scripts/dedup_stage_b_retrieval.py --paths configs/paths.yaml

In [ ]:
!cd /content/QAHERRA && python scripts/dedup_stage_c_prompts.py --paths configs/paths.yaml

In [ ]:
!cd /content/QAHERRA && python scripts/dedup_stage_d_qa.py --paths configs/paths.yaml

## 7. Figure 3

Drawn from `results/units_vs_chunks_tests.csv`, so the bars and the
significance brackets are the same numbers as Table III.

In [ ]:
!cd /content/QAHERRA && python scripts/make_figure3.py --out fig/fig3_representation_scope.pdf
from IPython.display import Image, display
display(Image('/content/QAHERRA/fig/fig3_representation_scope.png'))